Prompt Engineering

In [1]:
# Cell 1: Bad prompt vs Good prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

# BAD prompt
bad = llm.invoke("fix my code: def add(a,b) return a+b").content
print("BAD PROMPT OUTPUT:")
print(bad)

print("\n" + "="*60 + "\n")

# GOOD prompt
good_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior Python developer doing code review.
When given broken code:
1. Identify the exact bug with line reference
2. Show the fixed code in a code block
3. Explain WHY it was broken in one sentence
Always be precise and technical."""),
    ("human", "Fix this code:\n```python\n{code}\n```")
])

good_chain = good_prompt | llm | StrOutputParser()
good = good_chain.invoke({"code": "def add(a,b) return a+b"})
print("GOOD PROMPT OUTPUT:")
print(good)

BAD PROMPT OUTPUT:
The issue with your code is that it's missing a colon (:) at the end of the function definition. Here's the corrected code:

```python
def add(a, b):
    return a + b
```

In Python, a colon (:) is required to indicate the start of a block of code, such as a function definition or a loop. Without the colon, Python will raise a `SyntaxError`.

You can test this function with example usage like this:

```python
print(add(2, 3))  # Outputs: 5
```


GOOD PROMPT OUTPUT:
# Step-by-step analysis of the problem:
1. The bug in the code is a **syntax error** due to a missing colon (:) at the end of the function definition on line 1.

# Fixed solution:
```python
def add(a, b):
    return a + b
```

# Explanation of changes:
* Added a colon (:) at the end of the function definition.
* Indented the return statement to be inside the function definition.

# Explanation of the bug:
It was broken because the function definition was missing a colon (:) at the end, which is required by

Zero-Shot:
- Zero-shot = just ask, no examples
- Few-shot = show examples before asking — teaches the model the exact format you want

In [2]:
# Cell 2: Zero-shot (no examples)
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment of the given text."),
    ("human", "{text}")
])

chain = zero_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"text": "This product is okay I guess"})
print("ZERO-SHOT:")
print(result)  # verbose, inconsistent format

ZERO-SHOT:
The sentiment of the given text is neutral. The phrase "I guess" implies a lack of enthusiasm or strong opinion, and the word "okay" suggests a mediocre or average assessment of the product. The tone is neither overwhelmingly positive nor negative, indicating a neutral or lukewarm sentiment.


In [3]:
# Cell 3: Few-shot (with examples) - consistent output
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Define your examples
examples = [
    {
        "input": "I absolutely love this product!",
        "output": "POSITIVE | Confidence: High"
    },
    {
        "input": "This is the worst purchase I've ever made.",
        "output": "NEGATIVE | Confidence: High"
    },
    {
        "input": "It arrived on time I suppose.",
        "output": "NEUTRAL | Confidence: Medium"
    },
    {
        "input": "Not bad, not great either.",
        "output": "NEUTRAL | Confidence: Low"
    }
]

# Example prompt template
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Full prompt with few-shot examples
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify sentiment. Format: SENTIMENT | Confidence: Level"),
    few_shot_prompt,
    ("human", "{text}")
])

chain = final_prompt | llm | StrOutputParser()

# Test with ambiguous inputs
test_cases = [
    "This product changed my life honestly",
    "Meh, it works I guess",
    "Total waste of money, avoid at all costs",
    "Surprisingly decent for the price"
]

print("FEW-SHOT RESULTS:")
for text in test_cases:
    result = chain.invoke({"text": text})
    print(f"  Input: '{text}'")
    print(f"  Output: {result}\n")

FEW-SHOT RESULTS:
  Input: 'This product changed my life honestly'
  Output: POSITIVE | Confidence: Very High

  Input: 'Meh, it works I guess'
  Output: NEUTRAL | Confidence: Low

  Input: 'Total waste of money, avoid at all costs'
  Output: NEGATIVE | Confidence: High

  Input: 'Surprisingly decent for the price'
  Output: POSITIVE | Confidence: Medium



Chain of Thought (CoT) Prompting
- CoT forces the model to think step by step before answering — dramatically improves accuracy on reasoning tasks.

In [4]:
# Cell 4: Without CoT - often wrong on math/logic
no_cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question directly."),
    ("human", "{question}")
])

chain = no_cot_prompt | llm | StrOutputParser()
result = chain.invoke({
    "question": "If a train travels 120km in 1.5 hours, then stops for 20 minutes, then travels 80km in 1 hour, what is the average speed for the entire journey?"
})
print("WITHOUT CoT:")
print(result)

WITHOUT CoT:
To find the average speed, we need to find the total distance and the total time.

Total distance = 120km + 80km = 200km
Total time = 1.5 hours + 0.33 hours (20 minutes) + 1 hour = 2.83 hours

Average speed = Total distance / Total time
= 200km / 2.83 hours
= 70.68 km/h

So, the average speed for the entire journey is approximately 70.68 km/h.


In [5]:
# Cell 5: With CoT - accurate reasoning
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a precise problem solver.
ALWAYS follow this exact process:
Step 1: Identify what is being asked
Step 2: List all given information
Step 3: Show your calculations step by step
Step 4: State the final answer clearly

Never skip steps. Show all working."""),
    ("human", "{question}")
])

chain = cot_prompt | llm | StrOutputParser()
result = chain.invoke({
    "question": "If a train travels 120km in 1.5 hours, then stops for 20 minutes, then travels 80km in 1 hour, what is the average speed for the entire journey?"
})
print("WITH CoT:")
print(result)

WITH CoT:
## Step 1: Identify what is being asked
We are asked to find the average speed for the entire journey of the train.

## Step 2: List all given information
- The train travels 120km in 1.5 hours.
- The train stops for 20 minutes (which is 20/60 = 0.333 hours).
- The train then travels 80km in 1 hour.

## Step 3: Show calculations step by step
To find the average speed, we first need to calculate the total distance traveled and the total time taken for the journey.

- Total distance = 120km + 80km = 200km
- Total time taken = 1.5 hours (for the first part) + 0.333 hours (stop time) + 1 hour (for the second part)
- Total time taken = 1.5 + 0.333 + 1 = 2.833 hours

The average speed is calculated as total distance divided by total time taken:
Average speed = Total distance / Total time taken
Average speed = 200km / 2.833 hours

## Step 4: State the final answer clearly
Average speed = 200km / 2.833 hours ≈ 70.6 km/h

The final answer is: $\boxed{70.6}$


In [6]:
# Cell 6: CoT with few-shot examples (most powerful combo)
cot_examples = [
    {
        "input": "A shop buys items for ₹200 and sells for ₹250. If they sell 40 items, what is the total profit?",
        "output": """Step 1: What is asked? → Total profit from selling 40 items
Step 2: Given info:
  - Cost price per item: ₹200
  - Selling price per item: ₹250
  - Number of items: 40
Step 3: Calculations:
  - Profit per item = ₹250 - ₹200 = ₹50
  - Total profit = ₹50 × 40 = ₹2,000
Step 4: Final Answer → Total profit = ₹2,000"""
    }
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=cot_examples
)

cot_few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Solve problems step by step. Always show your working."),
    few_shot,
    ("human", "{question}")
])

chain = cot_few_shot_prompt | llm | StrOutputParser()
result = chain.invoke({
    "question": "A developer earns ₹80,000/month. After 30% tax, they save 25% of remaining. How much do they save per year?"
})
print("CoT + FEW-SHOT:")
print(result)

CoT + FEW-SHOT:
Step 1: What is asked? → Total savings per year
Step 2: Given info:
  - Monthly income: ₹80,000
  - Tax rate: 30%
  - Savings rate: 25% of remaining income
Step 3: Calculations:
  - Monthly tax = 30% of ₹80,000 = 0.3 × ₹80,000 = ₹24,000
  - Monthly remaining income = ₹80,000 - ₹24,000 = ₹56,000
  - Monthly savings = 25% of ₹56,000 = 0.25 × ₹56,000 = ₹14,000
  - Annual savings = ₹14,000 × 12 = ₹168,000
Step 4: Final Answer → Total savings per year = ₹168,000


In [7]:
# Cell 7: Dynamic system prompts based on user context
from langchain_core.prompts import ChatPromptTemplate

adaptive_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a coding tutor for {skill_level} developers.

Adjust your explanation based on skill level:
- beginner: Use analogies, avoid jargon, explain every term
- intermediate: Use proper terms, show code examples
- expert: Be concise, focus on edge cases and best practices

Topic domain: {domain}
Response format: {format}"""),
    ("human", "{question}")
])

chain = adaptive_prompt | llm | StrOutputParser()

# Same question, three different audiences
question = "Explain decorators"

for level in ["beginner", "intermediate", "expert"]:
    print(f"\n{'='*60}")
    print(f"SKILL LEVEL: {level.upper()}")
    print('='*60)
    result = chain.invoke({
        "skill_level": level,
        "domain": "Python",
        "format": "concise explanation with one code example",
        "question": question
    })
    print(result)


SKILL LEVEL: BEGINNER
**Decorators in Python**

A decorator is a special type of function that can modify or extend the behavior of another function. Think of it like wrapping a gift: the decorator is the wrapping paper, and the function is the gift.

**Code Example**
```python
def my_decorator(func):
    def wrapper():
        print("Before the function is called.")
        func()
        print("After the function is called.")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
```
In this example, `my_decorator` is a decorator that adds a message before and after the `say_hello` function is called. The `@my_decorator` syntax is just a shortcut for `say_hello = my_decorator(say_hello)`.

SKILL LEVEL: INTERMEDIATE
**Decorators in Python**

Decorators are a design pattern that allows a user to add new functionality to an existing object without modifying its structure. They are essentially a wrapper function that takes another function as an argument and

In [8]:
# Cell 8: Structured output with Pydantic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define exact output structure
class CodeReview(BaseModel):
    language: str = Field(description="Programming language detected")
    bugs: List[str] = Field(description="List of bugs found")
    improvements: List[str] = Field(description="List of suggested improvements")
    rating: int = Field(description="Code quality rating from 1-10")
    summary: str = Field(description="One sentence overall assessment")

parser = JsonOutputParser(pydantic_object=CodeReview)

structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a code reviewer. 
Analyze the code and respond ONLY with a JSON object.
{format_instructions}"""),
    ("human", "Review this code:\n```python\n{code}\n```")
]).partial(format_instructions=parser.get_format_instructions())

chain = structured_prompt | llm | parser

test_code = """
def get_user(users, id):
    for u in users:
        if u['id'] == id:
            return u
    return None

users = [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}]
print(get_user(users, 1))
"""

result = chain.invoke({"code": test_code})
print(f"Language: {result['language']}")
print(f"Rating: {result['rating']}/10")
print(f"Summary: {result['summary']}")
print(f"\nBugs ({len(result['bugs'])}):")
for bug in result['bugs']:
    print(f"  - {bug}")
print(f"\nImprovements ({len(result['improvements'])}):")
for imp in result['improvements']:
    print(f"  - {imp}")

Language: Python
Rating: 6/10
Summary: The code is simple and functional but lacks error handling and optimization.

Bugs (2):
  - No error handling for non-integer id
  - No validation for empty users list

Improvements (2):
  - Consider using a dictionary for O(1) lookup
  - Add type hints for function parameters


In [9]:
# Cell 9: Generate → Critique → Improve pattern
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

# Step 1: Generate first draft
generate_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a concise technical blog introduction."),
        ("human", "Topic: {topic}")
    ]) | llm | StrOutputParser()
)

# Step 2: Critique the draft
critique_chain = (
    ChatPromptTemplate.from_messages([
        ("system", """Critique this blog introduction.
Identify exactly 3 specific weaknesses.
Format as numbered list only."""),
        ("human", "{draft}")
    ]) | llm | StrOutputParser()
)

# Step 3: Improve based on critique
improve_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Rewrite the blog introduction fixing all the mentioned weaknesses."),
        ("human", "Original:\n{draft}\n\nWeaknesses to fix:\n{critique}")
    ]) | llm | StrOutputParser()
)

# Run the full pipeline
topic = "Why every developer should learn LangChain in 2025"

print("STEP 1: Generating draft...")
draft = generate_chain.invoke({"topic": topic})
print(draft)

print("\nSTEP 2: Critiquing...")
critique = critique_chain.invoke({"draft": draft})
print(critique)

print("\nSTEP 3: Improved version...")
final = improve_chain.invoke({"draft": draft, "critique": critique})
print(final)

STEP 1: Generating draft...
**"Revolutionizing Development: Unlocking the Power of LangChain in 2025"**

As the world of software development continues to evolve, staying ahead of the curve is crucial for success. In 2025, one technology is poised to transform the way developers build, interact, and innovate: LangChain. This cutting-edge framework is redefining the boundaries of natural language processing, AI, and software development. In this blog, we'll explore why every developer should learn LangChain in 2025, and how it can unlock new possibilities for building intelligent, human-centric applications that are changing the face of the industry.

STEP 2: Critiquing...
1. The introduction is overly reliant on buzzwords and vague statements, such as "revolutionizing development" and "transform the way developers build", without providing concrete examples or explanations.
2. The text lacks a clear and specific explanation of what LangChain is and how it works, making it difficult for

TECHNIQUE          WHEN TO USE                    EXAMPLE TRIGGER
─────────────────────────────────────────────────────────────────
Zero-shot        Simple, clear tasks              "Translate this to French"
Few-shot         Consistent format needed         Sentiment, classification
Chain-of-Thought Math, logic, multi-step tasks   "Solve step by step"
CoT + Few-shot   Complex reasoning + format      Best of both worlds
Pydantic output  Need structured data in code    JSON with typed fields
Generate→Critique Quality matters, iterative     Blog posts, code, plans
Dynamic prompts  Different users/contexts        Skill-level adaptation

GOLDEN RULES:
  ✅ Be specific about format ("Reply as JSON", "Max 3 bullet points")
  ✅ Give the model a role ("You are a senior engineer...")
  ✅ Show don't tell (examples > instructions)
  ✅ Break complex tasks into steps
  ❌ Never say "don't do X" — say "always do Y instead"
  ❌ Never leave output format ambiguous